In [191]:
import pandas as pd

In [192]:
df = pd.read_csv("../data/raw/bookings.csv")
df.head()

,key_id,booking_id,tournament_id,tournament_name,match_id,match_name,match_date,stage_name,group_name,team_id,...,given_name,shirt_number,minute_label,minute_regulation,minute_stoppage,match_period,yellow_card,red_card,second_yellow_card,sending_off
0,1,B-0001,WC-1970,1970 FIFA Men's World Cup,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,Group 1,T-72,...,Kakhi,11,30',30,0,first half,1,0,0,0
1,2,B-0002,WC-1970,1970 FIFA Men's World Cup,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,Group 1,T-72,...,Givi,19,31',31,0,first half,1,0,0,0
2,3,B-0003,WC-1970,1970 FIFA Men's World Cup,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,Group 1,T-72,...,Evgeny,6,34',34,0,first half,1,0,0,0
3,4,B-0004,WC-1970,1970 FIFA Men's World Cup,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,Group 1,T-46,...,Gustavo,3,60',60,0,second half,1,0,0,0
4,5,B-0005,WC-1970,1970 FIFA Men's World Cup,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,Group 1,T-72,...,Gennady,7,72',72,0,second half,1,0,0,0


In [193]:
print(f"Raw data shape: {df.shape}\n")
print(f"Columns: \n{df.columns}\n")
print(f"Misisng values: \n{df.isnull().sum()}")

Raw data shape: (3178, 26)

Columns: 
Index(['key_id', 'booking_id', 'tournament_id', 'tournament_name', 'match_id',
       'match_name', 'match_date', 'stage_name', 'group_name', 'team_id',
       'team_name', 'team_code', 'home_team', 'away_team', 'player_id',
       'family_name', 'given_name', 'shirt_number', 'minute_label',
       'minute_regulation', 'minute_stoppage', 'match_period', 'yellow_card',
       'red_card', 'second_yellow_card', 'sending_off'],
      dtype='object')

Misisng values: 
key_id                0
booking_id            0
tournament_id         0
tournament_name       0
match_id              0
match_name            0
match_date            0
stage_name            0
group_name            0
team_id               0
team_name             0
team_code             0
home_team             0
away_team             0
player_id             0
family_name           0
given_name            0
shirt_number          0
minute_label          0
minute_regulation     0
minute_stoppag

In [194]:
# This work focuses on FIFA Men's World Cup only

df = df[df["tournament_name"].str.contains("Men's")]
print(f"Current data shape: {df.shape}")

Current data shape: (2624, 26)


In [195]:
# drop ununsed/irrelevant columns

df = df.drop(columns = ["key_id", "tournament_name", "group_name", "team_code", 
                        "shirt_number", "family_name", "given_name", "minute_regulation", "home_team", 
                        "away_team", "minute_label", "minute_stoppage", "match_period", "sending_off"])
print(f"Current data shape: {df.shape} \n")
print(f"Columns: {df.columns}")
df.head()

Current data shape: (2624, 12) 

Columns: Index(['booking_id', 'tournament_id', 'match_id', 'match_name', 'match_date',
       'stage_name', 'team_id', 'team_name', 'player_id', 'yellow_card',
       'red_card', 'second_yellow_card'],
      dtype='object')


,booking_id,tournament_id,match_id,match_name,match_date,stage_name,team_id,team_name,player_id,yellow_card,red_card,second_yellow_card
0,B-0001,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,P-33189,1,0,0
1,B-0002,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,P-43733,1,0,0
2,B-0003,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,P-52906,1,0,0
3,B-0004,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-46,Mexico,P-64553,1,0,0
4,B-0005,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,P-70408,1,0,0


In [196]:
# load the team dataset - confederation lookup 

df_teams = pd.read_csv("../data/raw/teams.csv")
df_teams = df_teams[["team_id", "team_name", "confederation_code"]]
df_teams = df_teams[["team_id", "confederation_code"]]
df_teams.head()

,team_id,confederation_code
0,T-01,CAF
1,T-02,CAF
2,T-03,CONMEBOL
3,T-04,AFC
4,T-05,UEFA


In [197]:
# referee dataset - referee confederation lookup

df_ref = pd.read_csv("../data/processed/referee.csv")
df_ref = df_ref[["match_id", "referee_id", "confederation_code"]]
df_ref = df_ref.rename(columns = {"confederation_code": "ref_confederation"})
df_ref.head()

,match_id,referee_id,ref_confederation
0,M-1970-01,R-440,UEFA
1,M-1970-02,R-115,UEFA
2,M-1970-03,R-254,UEFA
3,M-1970-04,R-392,UEFA
4,M-1970-05,R-358,UEFA


In [198]:
df = pd.merge(
    df, 
    df_ref, 
    on = "match_id", 
    how = "left"
).merge(
    df_teams.rename(columns = {"confederation_code": "team_confederation"}), 
    on = "team_id",
    how = "left"
)

desired_order = ['booking_id', 'tournament_id', 'match_id', 'match_name', 'match_date',
                'stage_name', 'team_id', 'team_name','team_confederation', 'player_id',
                'yellow_card', 'red_card', 'second_yellow_card', 'referee_id', 'ref_confederation']
df = df.reindex(columns = desired_order)

print(f"Current data shape: {df.shape} \n")
print(f"Columns: {df.columns}")
df.head()

Current data shape: (2624, 15) 

Columns: Index(['booking_id', 'tournament_id', 'match_id', 'match_name', 'match_date',
       'stage_name', 'team_id', 'team_name', 'team_confederation', 'player_id',
       'yellow_card', 'red_card', 'second_yellow_card', 'referee_id',
       'ref_confederation'],
      dtype='object')


,booking_id,tournament_id,match_id,match_name,match_date,stage_name,team_id,team_name,team_confederation,player_id,yellow_card,red_card,second_yellow_card,referee_id,ref_confederation
0,B-0001,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,UEFA,P-33189,1,0,0,R-440,UEFA
1,B-0002,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,UEFA,P-43733,1,0,0,R-440,UEFA
2,B-0003,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,UEFA,P-52906,1,0,0,R-440,UEFA
3,B-0004,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-46,Mexico,CONCACAF,P-64553,1,0,0,R-440,UEFA
4,B-0005,WC-1970,M-1970-01,Mexico vs Soviet Union,1970-05-31,group stage,T-72,Soviet Union,UEFA,P-70408,1,0,0,R-440,UEFA


In [199]:
df.groupby(["yellow_card", "red_card", "second_yellow_card"]).size().reset_index()

,yellow_card,red_card,second_yellow_card,0
0,0,1,0,76
1,1,0,0,2470
2,1,0,1,62
3,1,1,0,16


In [202]:
def reset_yellow(row):
    if row["red_card"] == 1:
        return 0
    else:
        return 1

In [203]:
df["yellow_card"] = df.apply(reset_yellow, axis = 1)

In [204]:
df.groupby(["yellow_card", "red_card", "second_yellow_card"]).size().reset_index()

,yellow_card,red_card,second_yellow_card,0
0,0,1,0,92
1,1,0,0,2470
2,1,0,1,62


In [205]:
df["yellow_card"] = df["yellow_card"] + df["second_yellow_card"]

In [206]:
df.groupby(["yellow_card", "red_card", "second_yellow_card"]).size().reset_index()

,yellow_card,red_card,second_yellow_card,0
0,0,1,0,92
1,1,0,0,2470
2,2,0,1,62
